In [ ]:
import scanpy as sc

In [ ]:
adata = sc.read_h5ad("./data/preprocessed_replogle_k562.h5ad")

In [ ]:
adata

AnnData object with n_obs × n_vars = 162751 × 5657
    obs: 'condition', 'cell_type', 'dose_val', 'control', 'condition_name'
    var: 'chr', 'start', 'end', 'class', 'strand', 'length', 'in_matrix', 'mean', 'std', 'cv', 'fano', 'ensembl_id', 'ncounts', 'ncells', 'gene_name', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'hvg', 'log1p', 'neighbors', 'non_dropout_gene_idx', 'non_zeros_gene_idx', 'pca', 'rank_genes_groups_cov_all', 'rank_genes_groups_list', 'top_non_dropout_de_20', 'top_non_zero_de_20', 'umap'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    layers: 'counts'
    obsp: 'connectivities', 'distances'

In [ ]:
adata.obs['condition'].value_counts()

condition
ctrl            10691
NCBP2+ctrl        765
SLC39A9+ctrl      724
DONSON+ctrl       688
GAB2+ctrl         637
                ...  
BMS1+ctrl          18
SF1+ctrl           18
RPS15+ctrl         18
SRSF3+ctrl         16
BUD31+ctrl         15
Name: count, Length: 1093, dtype: int64

In [ ]:
adata.obs['condition'].value_counts().shape

(1093,)

In [ ]:
sel_ind = [0, 1, 2, 100, 300, 600, 900, 1000, 1090, 1092]
sel_cond = adata.obs['condition'].value_counts().index[sel_ind].tolist()
adata_small = adata[adata.obs['condition'].isin(sel_cond)].copy()

In [ ]:
# subset to 1000 ctrl cells
ctrl_idx = adata_small.obs.loc[
    adata_small.obs['condition'] == 'ctrl'
].sample(
    n=min(1000, (adata_small.obs['condition'] == 'ctrl').sum()),
    random_state=42
).index

adata_small = adata_small[
    (adata_small.obs['condition'] != 'ctrl') | adata_small.obs.index.isin(ctrl_idx)
].copy()

In [ ]:
adata_small.obs['condition'].value_counts()

condition
ctrl            1000
NCBP2+ctrl       765
SLC39A9+ctrl     724
RPS28+ctrl       258
TSEN2+ctrl       168
NOL6+ctrl        107
SKP1+ctrl         64
ACTL6A+ctrl       45
RPS15+ctrl        18
BUD31+ctrl        15
Name: count, dtype: int64

In [ ]:
import pickle

split_dict = {
    'train': ['ctrl', 'NCBP2+ctrl', 'RPS28+ctrl', 'TSEN2+ctrl', 'ACTL6A+ctrl', 'RPS15+ctrl'],
    'val': ['SKP1+ctrl'],
    'test': ['SLC39A9+ctrl', 'NOL6+ctrl', 'BUD31+ctrl']
}
pickle.dump(split_dict, open("./data/adata_replogle_k562_small_split_dict.pkl", "wb"))

In [ ]:
del adata_small.obsm
del adata_small.varm
del adata_small.layers
del adata_small.obsp

In [ ]:
adata_small

AnnData object with n_obs × n_vars = 3164 × 5657
    obs: 'condition', 'cell_type', 'dose_val', 'control', 'condition_name'
    var: 'chr', 'start', 'end', 'class', 'strand', 'length', 'in_matrix', 'mean', 'std', 'cv', 'fano', 'ensembl_id', 'ncounts', 'ncells', 'gene_name', 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'hvg', 'log1p', 'neighbors', 'non_dropout_gene_idx', 'non_zeros_gene_idx', 'pca', 'rank_genes_groups_cov_all', 'rank_genes_groups_list', 'top_non_dropout_de_20', 'top_non_zero_de_20', 'umap'

In [ ]:
adata_small.write_h5ad("./data/preprocessed_replogle_k562_small.h5ad")